[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Testing and Packaging](https://johnfisher-ai.github.io/Python-Visual-Guides/testing-and-packaging.html)

# Parametrize


## What you will be able to do

Run one test over many inputs with `@pytest.mark.parametrize`, every input reported as a test of its
own, with an ID that says which case it is. Give cases IDs you can read, mark a case that is expected
to fail, combine parameters into every combination, give a fixture a list of values, and run a single
case by its node ID.


## The idea

### The problem

`to_fahrenheit` deserves checking at more than one temperature: freezing, boiling, -40, where the
two scales agree, and body temperature. Following the **Test Structure** notebook, that makes four
tests of one behavior each, four functions that differ only in two numbers, and a fifth temperature
means a fifth copy. A loop inside one test is shorter, and it stops at the first temperature that
fails, so its report names one case and says nothing about the others.

`parse_reading` has the same problem with more cases: a plain line, an empty reading, a Unicode minus
sign, spaces around the line, and every odd line the stations send next month. The check is the same
every time, and only the data changes, so the data is what a test file should make easy to add to.

### What parametrizing a test is

> **Parametrizing** a test runs it once for each set of values in a list.
> `@pytest.mark.parametrize("celsius, fahrenheit", [(0, 32), (100, 212)])` names the test's
> arguments and gives a set of values for every run, and pytest collects each run as a test of its
> own, reported on its own, with a **test ID** in square brackets after the test's name. pytest
> builds an ID from the values, or takes it from `ids`, or from `pytest.param`, which also lets one
> set of values carry a **mark**, such as `xfail` for a case that is expected to fail. Stacked
> `parametrize` decorators run every combination of their values, and a fixture given `params` runs
> every test that requests it once for each value.

### Why it works that way

- **The test is written once, and the table grows.** Adding a case adds one line of values, and no
  code.
- **Every case is a test.** A case has its own line in `-v`, its own node ID and its own failure, so
  one broken case hides no other, and `--lf` runs only the cases that failed.
- **An ID is built from the values when it can be.** Numbers and short strings make IDs such as
  `[37-98.6]`. A tuple or a list becomes the argument's name and a count, such as `expected2`, which
  `ids` and `pytest.param` replace with words.
- **`@pytest.mark.parametrize` is a mark.** It attaches the values to the function and returns the
  function unchanged, and pytest reads the mark when it collects the test. This is the marking the
  **Decorators** notebook described.
- **Stacking multiplies.** Three stations and three temperatures make nine tests, which is thorough,
  and a suite built that way grows slow.
- **A parametrized fixture spreads its cases.** Every test that requests it runs once for each value,
  so a new kind of input reaches all of those tests at once.

### Where you will meet this

pytest's documentation introduces parametrize with a table of expressions and their values, such as
`("3+5", 8)`, and marks the case that fails with `xfail`. A parser or a converter is often tested
this way, by a table of inputs and results. The **Encodings** notebook in the **Files, Paths and
Formats** guide showed the byte order mark that Excel writes at the start of a UTF-8 file, and this
notebook's parametrized fixture tests a file with one and a file without. The **Testing Failure**
notebook parametrizes the lines `parse_reading` must reject.

### What this notebook covers

- One test over many inputs, with `@pytest.mark.parametrize`
- A failing case, reported alone, and one case run by its node ID
- IDs you can read, with `ids` and with `pytest.param`
- A case expected to fail, marked with `xfail`
- Stacked decorators, and every combination of their values
- A parametrized fixture, which runs every test that requests it once for each value
- A day's file, tested as a table of cases in two encodings
- Four errors: a count that does not match, a misspelled argument, tuples for a single name, and a
  loop that hides failures

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import subprocess
import sys
from pathlib import Path

Path("test_first_look.py").write_text('''
import pytest


def to_fahrenheit(celsius):
    return celsius * 9 / 5 + 32


@pytest.mark.parametrize("celsius, fahrenheit", [(0, 32), (100, 212), (-40, -40), (37, 99)])
def test_to_fahrenheit(celsius, fahrenheit):
    assert to_fahrenheit(celsius) == fahrenheit
''')

finished = subprocess.run([sys.executable, "-m", "pytest", "-v", "--tb=no", "--no-header"],
                          capture_output=True, text=True)
print(finished.stdout.strip())
```

```
============================= test session starts ==============================
collecting ... collected 4 items

test_first_look.py::test_to_fahrenheit[0-32] PASSED                      [ 25%]
test_first_look.py::test_to_fahrenheit[100-212] PASSED                   [ 50%]
test_first_look.py::test_to_fahrenheit[-40--40] PASSED                   [ 75%]
test_first_look.py::test_to_fahrenheit[37-99] FAILED                     [100%]

=========================== short test summary info ============================
FAILED test_first_look.py::test_to_fahrenheit[37-99] - assert 98.6 == 99
========================= 1 failed, 3 passed in 0.01s ==========================
```

The test is written once, and the list beside it holds four cases. pytest ran it four times, reported
every run on a line of its own, and named each run by its values in brackets. The case with the wrong
value, 99 where 37 °C is 98.6 °F, failed alone, and the other three still passed.


## Setup

Six imports, and the functions that run pytest.

- `subprocess` runs pytest as a program of its own, in `run_pytest`
- `sys` names the Python that runs it
- `os` passes pytest the environment, with `NO_COLOR` and `PYTHONDONTWRITEBYTECODE` set in it, as in
  the **Your First Test** notebook
- `re` takes out of pytest's report the parts that differ between computers
- `Path` makes the project's folders, and checks what the tests leave behind
- `shutil` removes the scratch folder at the end

`pytest_report` and `run_pytest` are the functions the **Your First Test** notebook wrote, which run
`python -m pytest` in the project's folder, `scratch/stations`, and that notebook explains each of
their settings.


In [1]:
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

PROJECT = Path("scratch/stations")
(PROJECT / "tests").mkdir(parents=True, exist_ok=True)
os.environ["NO_COLOR"] = "1"                  # programs started from here print without color codes
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"   # and keep no compiled copies, which a quick rewrite can outrun


def pytest_report(*arguments, folder=PROJECT):
    """What python -m pytest prints when it runs in the folder, less what differs between computers."""
    settings = {"COLUMNS": "80", "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1", "PYTHONNODEBUGRANGES": "1"}
    finished = subprocess.run([sys.executable, "-m", "pytest", "--no-header", *arguments],
                              cwd=folder, capture_output=True, text=True, env={**os.environ, **settings})
    report = finished.stdout + finished.stderr
    report = report.replace(f"{Path(folder).resolve()}/", "")          # the folder's own path
    report = re.sub(r"\S*/_pytest/", "_pytest/", report)                # the path to pytest's own files
    return re.sub(r" in \d+\.\d+s\b", "", report).rstrip()              # the time the run took


def run_pytest(*arguments, folder=PROJECT):
    """Run python -m pytest in the folder, as a terminal would, and print its report."""
    print(pytest_report(*arguments, folder=folder))


print("ready:", PROJECT)


ready: scratch/stations


## Worked examples

### One test, many inputs

The module is the one the **Fixtures** notebook finished with, and one function joins it,
`summarize_file`, which opens a file of readings and summarizes it:


In [2]:
%%writefile scratch/stations/readings.py
"""Readings from the weather stations, and each station's mean temperature."""

import statistics


def parse_reading(line):
    """A (station, celsius) pair from a line such as 'Bergen,4.2'. An empty reading is None.

    A minus sign written as U+2212, as some spreadsheets write it, reads as a hyphen-minus.
    """
    station, celsius = line.strip().split(",")
    celsius = celsius.replace("\u2212", "-")
    return station, float(celsius) if celsius else None


def mean(values):
    """The mean of the readings that are not None, or None when there are none."""
    present = [value for value in values if value is not None]
    return statistics.fmean(present) if present else None


def summarize(lines):
    """Each station's mean temperature, from lines of readings. A blank line is skipped."""
    by_station = {}
    for line in lines:
        if not line.strip():
            continue
        station, celsius = parse_reading(line)
        by_station.setdefault(station, []).append(celsius)
    return {station: mean(values) for station, values in by_station.items()}


def summarize_file(path):
    """Each station's mean temperature, from a file of readings."""
    with open(path, encoding="utf-8") as file:
        return summarize(file)


def to_fahrenheit(celsius):
    """A temperature in degrees Celsius, in degrees Fahrenheit."""
    return celsius * 9 / 5 + 32


Writing scratch/stations/readings.py


`@pytest.mark.parametrize` takes the names of the test's arguments, as one string with commas between
them, and a list with a tuple of values for every run. The test itself is written once, and reads
the values from its arguments:


In [3]:
%%writefile scratch/stations/tests/test_conversions.py
import pytest

from readings import to_fahrenheit


@pytest.mark.parametrize("celsius, fahrenheit", [(0, 32), (100, 212), (-40, -40), (37, 98.6)])
def test_to_fahrenheit(celsius, fahrenheit):
    assert to_fahrenheit(celsius) == pytest.approx(fahrenheit)


Writing scratch/stations/tests/test_conversions.py


In [4]:
run_pytest("tests/test_conversions.py", "-v")


============================= test session starts ==============================
collecting ... collected 4 items

tests/test_conversions.py::test_to_fahrenheit[0-32] PASSED               [ 25%]
tests/test_conversions.py::test_to_fahrenheit[100-212] PASSED            [ 50%]
tests/test_conversions.py::test_to_fahrenheit[-40--40] PASSED            [ 75%]
tests/test_conversions.py::test_to_fahrenheit[37-98.6] PASSED            [100%]

============================== 4 passed ===============================


One function, and four tests. pytest made an ID for every case from its values, joined with `-`, so
`[-40--40]` is -40 and -40. `pytest.approx` covers 98.6, which is not exact in binary.

### A failing case, reported alone

Now somebody rounds the conversion, to print whole degrees. Three of the four temperatures convert
to whole degrees anyway, and body temperature does not. The cell keeps the module's text, to put it
back:


In [5]:
module = (PROJECT / "readings.py").read_text()          # the module as it is, to put back later
(PROJECT / "readings.py").write_text(module.replace("return celsius * 9 / 5 + 32", "return round(celsius * 9 / 5) + 32"))

run_pytest("tests/test_conversions.py", "-q")


...F                                                                     [100%]
=================================== FAILURES ===================================
_________________________ test_to_fahrenheit[37-98.6] __________________________

celsius = 37, fahrenheit = 98.6

    @pytest.mark.parametrize("celsius, fahrenheit", [(0, 32), (100, 212), (-40, -40), (37, 98.6)])
    def test_to_fahrenheit(celsius, fahrenheit):
>       assert to_fahrenheit(celsius) == pytest.approx(fahrenheit)
E       assert 99 == 98.6 ± 9.9e-05
E         
E         comparison failed
E         Obtained: 99
E         Expected: 98.6 ± 9.9e-05

tests/test_conversions.py:8: AssertionError
=========================== short test summary info ============================
FAILED tests/test_conversions.py::test_to_fahrenheit[37-98.6] - assert 99 == ...
1 failed, 3 passed


One case failed, and the three others passed and say so. The report's heading names the case,
`test_to_fahrenheit[37-98.6]`, and its first lines give the arguments that case received,
`celsius = 37, fahrenheit = 98.6`. The case's node ID runs it alone, and `-k` matches part of an ID
too, as it matches part of a name:


In [6]:
run_pytest("tests/test_conversions.py::test_to_fahrenheit[37-98.6]", "-q", "--tb=no")
print()
run_pytest("tests/test_conversions.py", "-k", "100", "-v")


F                                                                        [100%]
=========================== short test summary info ============================
FAILED tests/test_conversions.py::test_to_fahrenheit[37-98.6] - assert 99 == ...
1 failed

============================= test session starts ==============================
collecting ... collected 4 items / 3 deselected / 1 selected

tests/test_conversions.py::test_to_fahrenheit[100-212] PASSED            [100%]

======================= 1 passed, 3 deselected ========================


The rounding goes, and `--lf` runs only the case that failed:


In [7]:
(PROJECT / "readings.py").write_text(module)

run_pytest("tests/test_conversions.py", "--lf", "-v")


============================= test session starts ==============================
collecting ... collected 4 items / 3 deselected / 1 selected
run-last-failure: rerun previous 1 failure

tests/test_conversions.py::test_to_fahrenheit[37-98.6] PASSED            [100%]

======================= 1 passed, 3 deselected ========================


### IDs you can read

The IDs above are readable because the values are numbers. Here is `parse_reading` over a table of
lines, where every expected value is a tuple:


In [8]:
%%writefile scratch/stations/tests/test_parsing.py
import pytest

from readings import parse_reading


@pytest.mark.parametrize("line, expected", [
    ("Bergen,4.2", ("Bergen", 4.2)),
    ("Svalbard,", ("Svalbard", None)),
    ("Tromso,\u22126.3", ("Tromso", -6.3)),
    (" Oslo,-2.4 ", ("Oslo", -2.4)),
])
def test_parse_reading(line, expected):
    assert parse_reading(line) == expected


Writing scratch/stations/tests/test_parsing.py


In [9]:
run_pytest("tests/test_parsing.py", "--collect-only", "-q")


tests/test_parsing.py::test_parse_reading[Bergen,4.2-expected0]
tests/test_parsing.py::test_parse_reading[Svalbard,-expected1]
tests/test_parsing.py::test_parse_reading[Tromso,\u22126.3-expected2]
tests/test_parsing.py::test_parse_reading[ Oslo,-2.4 -expected3]

4 tests collected


A tuple has no short text of its own, so pytest used the argument's name and the case's position:
`expected0` to `expected3`. The line in each ID is readable, mostly: the Unicode minus sign appears
as the escape `\u2212`, and the fourth ID has spaces inside its brackets. `pytest.param` wraps one
case, and its `id` names it:


In [10]:
%%writefile scratch/stations/tests/test_parsing.py
import pytest

from readings import parse_reading


@pytest.mark.parametrize("line, expected", [
    pytest.param("Bergen,4.2", ("Bergen", 4.2), id="a reading"),
    pytest.param("Svalbard,", ("Svalbard", None), id="an empty reading"),
    pytest.param("Tromso,\u22126.3", ("Tromso", -6.3), id="a unicode minus sign"),
    pytest.param(" Oslo,-2.4 ", ("Oslo", -2.4), id="spaces around the line"),
])
def test_parse_reading(line, expected):
    assert parse_reading(line) == expected


Overwriting scratch/stations/tests/test_parsing.py


In [11]:
run_pytest("tests/test_parsing.py", "-v")


============================= test session starts ==============================
collecting ... collected 4 items

tests/test_parsing.py::test_parse_reading[a reading] PASSED              [ 25%]
tests/test_parsing.py::test_parse_reading[an empty reading] PASSED       [ 50%]
tests/test_parsing.py::test_parse_reading[a unicode minus sign] PASSED   [ 75%]
tests/test_parsing.py::test_parse_reading[spaces around the line] PASSED [100%]

============================== 4 passed ===============================


`ids`, a list with an ID for every case in order, does the same when the cases are short enough to
read without their names beside them:


In [12]:
%%writefile scratch/stations/tests/test_conversions.py
import pytest

from readings import to_fahrenheit


@pytest.mark.parametrize("celsius, fahrenheit", [(0, 32), (100, 212), (-40, -40), (37, 98.6)],
                         ids=["freezing", "boiling", "the same in both scales", "body temperature"])
def test_to_fahrenheit(celsius, fahrenheit):
    assert to_fahrenheit(celsius) == pytest.approx(fahrenheit)


Overwriting scratch/stations/tests/test_conversions.py


In [13]:
run_pytest("tests/test_conversions.py", "-v")


============================= test session starts ==============================
collecting ... collected 4 items

tests/test_conversions.py::test_to_fahrenheit[freezing] PASSED           [ 25%]
tests/test_conversions.py::test_to_fahrenheit[boiling] PASSED            [ 50%]
tests/test_conversions.py::test_to_fahrenheit[the same in both scales] PASSED [ 75%]
tests/test_conversions.py::test_to_fahrenheit[body temperature] PASSED   [100%]

============================== 4 passed ===============================


`pytest.param` keeps an ID beside its values, which suits a long table, where a separate list of IDs
would have to be kept in the same order by hand.

### A case expected to fail

A spreadsheet in a European locale writes a CSV with semicolons between fields and commas in numbers,
as the **CSV** notebook in the **Files, Paths and Formats** guide described, and `parse_reading` does
not read that yet. `marks=pytest.mark.xfail` records the case as expected to fail, with a reason, so
the table shows what is known not to work without making the suite fail:


In [14]:
%%writefile scratch/stations/tests/test_parsing.py
import pytest

from readings import parse_reading


@pytest.mark.parametrize("line, expected", [
    pytest.param("Bergen,4.2", ("Bergen", 4.2), id="a reading"),
    pytest.param("Svalbard,", ("Svalbard", None), id="an empty reading"),
    pytest.param("Tromso,\u22126.3", ("Tromso", -6.3), id="a unicode minus sign"),
    pytest.param(" Oslo,-2.4 ", ("Oslo", -2.4), id="spaces around the line"),
    pytest.param("Bergen;4,2", ("Bergen", 4.2), id="semicolons",
                 marks=pytest.mark.xfail(reason="semicolons are not read yet")),
])
def test_parse_reading(line, expected):
    assert parse_reading(line) == expected


Overwriting scratch/stations/tests/test_parsing.py


In [15]:
run_pytest("tests/test_parsing.py", "-v")


============================= test session starts ==============================
collecting ... collected 5 items

tests/test_parsing.py::test_parse_reading[a reading] PASSED              [ 20%]
tests/test_parsing.py::test_parse_reading[an empty reading] PASSED       [ 40%]
tests/test_parsing.py::test_parse_reading[a unicode minus sign] PASSED   [ 60%]
tests/test_parsing.py::test_parse_reading[spaces around the line] PASSED [ 80%]
tests/test_parsing.py::test_parse_reading[semicolons] XFAIL (semicol...) [100%]

========================= 4 passed, 1 xfailed =========================


`XFAIL` is not a pass and not a failure: the case ran and failed as expected, `-v` shows its reason,
shortened here to fit the line, and the counts list it as `xfailed`. A skipped test does not run at
all, and an `xfail` does, so the day `parse_reading` learns semicolons, the report shows that this
case passes.

### Every combination: stacked decorators

Two `parametrize` decorators on one test run it for every combination of their values. A reading
should come back from `parse_reading` as it went in, for every station and every temperature:


In [16]:
%%writefile scratch/stations/tests/test_round_trip.py
import pytest

from readings import parse_reading


@pytest.mark.parametrize("station", ["Bergen", "Oslo", "Tromso"])
@pytest.mark.parametrize("celsius", [-6.3, 0.0, 4.2])
def test_a_reading_comes_back_as_it_went_in(station, celsius):
    assert parse_reading(f"{station},{celsius}") == (station, celsius)


Writing scratch/stations/tests/test_round_trip.py


In [17]:
run_pytest("tests/test_round_trip.py", "-v")


============================= test session starts ==============================
collecting ... collected 9 items

tests/test_round_trip.py::test_a_reading_comes_back_as_it_went_in[-6.3-Bergen] PASSED [ 11%]
tests/test_round_trip.py::test_a_reading_comes_back_as_it_went_in[-6.3-Oslo] PASSED [ 22%]
tests/test_round_trip.py::test_a_reading_comes_back_as_it_went_in[-6.3-Tromso] PASSED [ 33%]
tests/test_round_trip.py::test_a_reading_comes_back_as_it_went_in[0.0-Bergen] PASSED [ 44%]
tests/test_round_trip.py::test_a_reading_comes_back_as_it_went_in[0.0-Oslo] PASSED [ 55%]
tests/test_round_trip.py::test_a_reading_comes_back_as_it_went_in[0.0-Tromso] PASSED [ 66%]
tests/test_round_trip.py::test_a_reading_comes_back_as_it_went_in[4.2-Bergen] PASSED [ 77%]
tests/test_round_trip.py::test_a_reading_comes_back_as_it_went_in[4.2-Oslo] PASSED [ 88%]
tests/test_round_trip.py::test_a_reading_comes_back_as_it_went_in[4.2-Tromso] PASSED [100%]

============================== 9 passed ===================

Three stations and three temperatures made nine tests. The ID starts with the temperature, from the
decorator nearer the function, and every combination appears once. Each decorator multiplies the
count, so a third with three values would make twenty-seven.

### A parametrized fixture: every test, for every kind of file

A fixture with `params` runs every test that requests it once for each value, which it reads from
`request.param`, where `request` is a fixture pytest provides. `tuesday_file` writes Tuesday's
readings twice: once as plain UTF-8, and once with the byte order mark Excel writes. The files go in
the folder the tests run in, and the `yield` removes each one:


In [18]:
%%writefile scratch/stations/tests/test_files.py
from pathlib import Path

import pytest

from readings import summarize_file

TUESDAY = ["Bergen,4.2", "Bergen,5.8", "Bergen,", "Oslo,-2.4", "Oslo,-1.6",
           "Svalbard,", "Svalbard,", "Tromso,-6.3", "Tromso,-5.7", "Tromso,"]


@pytest.fixture(params=["utf-8", "utf-8-sig"], ids=["utf-8", "utf-8 with a byte order mark"])
def tuesday_file(request):
    """Tuesday's readings in a file, written in each encoding in turn."""
    path = Path(f"tuesday-{request.param}.csv")
    path.write_text("\n".join(TUESDAY) + "\n", encoding=request.param)
    yield path
    path.unlink()


def test_every_station_is_summarized(tuesday_file):
    assert sorted(summarize_file(tuesday_file)) == ["Bergen", "Oslo", "Svalbard", "Tromso"]


def test_bergen_has_a_mean(tuesday_file):
    assert summarize_file(tuesday_file)["Bergen"] == 5.0


Writing scratch/stations/tests/test_files.py


In [19]:
run_pytest("tests/test_files.py", "-v")


============================= test session starts ==============================
collecting ... collected 4 items

tests/test_files.py::test_every_station_is_summarized[utf-8] PASSED      [ 25%]
tests/test_files.py::test_every_station_is_summarized[utf-8 with a byte order mark] FAILED [ 50%]
tests/test_files.py::test_bergen_has_a_mean[utf-8] PASSED                [ 75%]
tests/test_files.py::test_bergen_has_a_mean[utf-8 with a byte order mark] FAILED [100%]

=================================== FAILURES ===================================
________ test_every_station_is_summarized[utf-8 with a byte order mark] ________

tuesday_file = PosixPath('tuesday-utf-8-sig.csv')

    def test_every_station_is_summarized(tuesday_file):
>       assert sorted(summarize_file(tuesday_file)) == ["Bergen", "Oslo", "Svalbard", "Tromso"]
E       AssertionError: assert ['Bergen', 'O...\ufeffBergen'] == ['Bergen', 'O...rd', 'Tromso']
E         
E         Left contains one more item: '\ufeffBergen'
E         


Two tests and two encodings made four runs, and the byte order mark broke both tests. The mark sits
before the first line's first character, so `summarize_file` read that line's station as
`'\ufeffBergen'`, a different station from the `'Bergen'` on the next line. The second test's report
is the clearer of the two: Bergen's mean is 5.8, the one reading left on the lines without the mark.
`utf-8-sig`, which the **Encodings** notebook recommended for anything that may come from Excel,
reads a file with a byte order mark or without one:


In [20]:
module = module.replace('open(path, encoding="utf-8")', 'open(path, encoding="utf-8-sig")')
(PROJECT / "readings.py").write_text(module)

run_pytest("tests/test_files.py", "-v")
print("\nfiles left behind:", sorted(path.name for path in PROJECT.glob("tuesday-*.csv")))


============================= test session starts ==============================
collecting ... collected 4 items

tests/test_files.py::test_every_station_is_summarized[utf-8] PASSED      [ 25%]
tests/test_files.py::test_every_station_is_summarized[utf-8 with a byte order mark] PASSED [ 50%]
tests/test_files.py::test_bergen_has_a_mean[utf-8] PASSED                [ 75%]
tests/test_files.py::test_bergen_has_a_mean[utf-8 with a byte order mark] PASSED [100%]

============================== 4 passed ===============================

files left behind: []


### A day's file, as a table of cases in two encodings

The pieces of this notebook, in one test of `summarize_file`. A table holds the days, each with an
ID, and the one the module cannot read yet is marked `xfail`. A parametrized fixture gives every day
both encodings, and the test writes the day into `tmp_path` in the encoding it receives:


In [21]:
%%writefile scratch/stations/tests/test_days.py
import pytest

from readings import summarize_file

DAYS = [
    pytest.param(["Bergen,4.2", "Bergen,5.8"], {"Bergen": 5.0}, id="one station"),
    pytest.param(["Svalbard,", "Svalbard,"], {"Svalbard": None}, id="only empty readings"),
    pytest.param(["Tromso,\u22126.3", "", "Tromso,-5.7"], {"Tromso": -6.0}, id="a minus sign and a blank line"),
    pytest.param(["Bergen;4,2"], {"Bergen": 4.2}, id="semicolons",
                 marks=pytest.mark.xfail(reason="semicolons are not read yet")),
]


@pytest.fixture(params=["utf-8", "utf-8-sig"])
def encoding(request):
    return request.param


@pytest.mark.parametrize("lines, expected", DAYS)
def test_a_day_summarizes(lines, expected, encoding, tmp_path):
    path = tmp_path / "day.csv"
    path.write_text("\n".join(lines) + "\n", encoding=encoding)

    summary = summarize_file(path)

    assert summary == expected


Writing scratch/stations/tests/test_days.py


In [22]:
run_pytest("tests/test_days.py", "-v")
print()
run_pytest("-q")


============================= test session starts ==============================
collecting ... collected 8 items

tests/test_days.py::test_a_day_summarizes[utf-8-one station] PASSED      [ 12%]
tests/test_days.py::test_a_day_summarizes[utf-8-only empty readings] PASSED [ 25%]
tests/test_days.py::test_a_day_summarizes[utf-8-a minus sign and a blank line] PASSED [ 37%]
tests/test_days.py::test_a_day_summarizes[utf-8-semicolons] XFAIL (s...) [ 50%]
tests/test_days.py::test_a_day_summarizes[utf-8-sig-one station] PASSED  [ 62%]
tests/test_days.py::test_a_day_summarizes[utf-8-sig-only empty readings] PASSED [ 75%]
tests/test_days.py::test_a_day_summarizes[utf-8-sig-a minus sign and a blank line] PASSED [ 87%]
tests/test_days.py::test_a_day_summarizes[utf-8-sig-semicolons] XFAIL    [100%]

========================= 6 passed, 2 xfailed =========================

.......x...x........x.........                                           [100%]
27 passed, 3 xfailed


Four days and two encodings made eight tests: six passed, and the two semicolon cases failed as
expected. An ID joins the encoding and the day's name. A new day is one more line in `DAYS`, and it
is tested in both encodings without a change to the test.

### Where each part came from

| In the test | What it relies on | The section that showed it |
|---|---|---|
| `@pytest.mark.parametrize("lines, expected", DAYS)` | one test, run for every set of values | One test, many inputs |
| a failing day would fail alone | every case reported as a test of its own | A failing case, reported alone |
| `pytest.param(..., id="one station")` | an ID beside its values | IDs you can read |
| `marks=pytest.mark.xfail(reason=...)` | a known gap recorded without failing the suite | A case expected to fail |
| `test_a_day_summarizes[utf-8-sig-one station]` | two parameters, every combination | Every combination: stacked decorators |
| `encoding`, with `params` | a fixture that runs every test once for each value | A parametrized fixture: every test, for every kind of file |

The test's code is five lines, and everything that varies lives in `DAYS` and in the fixture's
`params`.


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/testing-and-packaging/05-parametrize-solutions.ipynb).

**1.** Write `tests/test_tasks.py` with a test of `mean`, parametrized over three cases: `[4.2, 5.8]`
gives `5.0`, `[None, -6.3]` gives `-6.3`, and `[None]` gives `None`. Run the file with `-v`, and read
the IDs pytest made.


In [23]:
# your code here


**2.** Give the three cases IDs with `ids`: `two readings`, `a missing reading` and `no readings`.
Run the file with `-v`.


In [24]:
# your code here


**3.** Run only the `no readings` case, by its node ID.


In [25]:
# your code here


**4.** Rewrite the cases with `pytest.param`, keeping their IDs, and add a fourth, with the ID
`a reading as text`: `["4.2"]` gives `4.2`, marked `xfail` with the reason
`readings arrive as numbers`. Run the file with `-v`.


In [26]:
# your code here


**5.** Write `tests/test_combinations.py` with a test parametrized by two stacked decorators,
stations `Bergen` and `Oslo`, and readings `"4.2"` and `""`, which checks that
`parse_reading(f"{station},{reading}")` returns the station first. Run it with `-q` and count the
tests.


In [27]:
# your code here


**6.** Write `tests/test_line_endings.py` with a fixture, `line_ending`, whose `params` are `"\n"`
and `"\r\n"`, with the IDs `unix` and `windows`, and a test that writes two readings from Oslo into
a file in `tmp_path` with that ending, and checks that `summarize_file` gives `{"Oslo": -2.0}`. Run
it with `-v`.


In [28]:
# your code here


## Common errors

### in "parametrize" the number of names (2)


In [29]:
%%writefile scratch/stations/tests/test_count.py
import pytest

from readings import to_fahrenheit


@pytest.mark.parametrize("celsius, fahrenheit", [(0, 32), (100,)])
def test_to_fahrenheit(celsius, fahrenheit):
    assert to_fahrenheit(celsius) == fahrenheit


Writing scratch/stations/tests/test_count.py


In [30]:
run_pytest("tests/test_count.py", "-q")



==================================== ERRORS ====================================
_____________________ ERROR collecting tests/test_count.py _____________________
tests/test_count.py::test_to_fahrenheit: in "parametrize" the number of names (2):
  ['celsius', 'fahrenheit']
must be equal to the number of values (1):
  (100,)
=========================== short test summary info ============================
ERROR tests/test_count.py - Failed: tests/test_count.py::test_to_fahrenheit: ...
!!!!!!!!!!!!!!!!!!!! Interrupted: 1 error during collection !!!!!!!!!!!!!!!!!!!!
1 error


Two names, and a case with one value, so pytest could not give `fahrenheit` anything, and it stopped
while collecting the file, before any test ran. The report shows the names and the case that is
short. Every case needs a value for every name:


In [31]:
source = (PROJECT / "tests" / "test_count.py").read_text()
(PROJECT / "tests" / "test_count.py").write_text(source.replace("(100,)", "(100, 212)"))

run_pytest("tests/test_count.py", "-q")


..                                                                       [100%]
2 passed


### In test_to_fahrenheit: function uses no argument 'celcius'


In [32]:
%%writefile scratch/stations/tests/test_spelling.py
import pytest

from readings import to_fahrenheit


@pytest.mark.parametrize("celcius, fahrenheit", [(0, 32), (100, 212)])
def test_to_fahrenheit(celsius, fahrenheit):
    assert to_fahrenheit(celsius) == fahrenheit


Writing scratch/stations/tests/test_spelling.py


In [33]:
run_pytest("tests/test_spelling.py", "-q")



==================================== ERRORS ====================================
___________________ ERROR collecting tests/test_spelling.py ____________________
In test_to_fahrenheit: function uses no argument 'celcius'
=========================== short test summary info ============================
ERROR tests/test_spelling.py - Failed: In test_to_fahrenheit: function uses n...
!!!!!!!!!!!!!!!!!!!! Interrupted: 1 error during collection !!!!!!!!!!!!!!!!!!!!
1 error


The names in the string must be the names of the test's arguments, and `celcius` is not one of them.
pytest matches them as text, so a misspelling in either place stops the collection. Spell both the
same:


In [34]:
source = (PROJECT / "tests" / "test_spelling.py").read_text()
(PROJECT / "tests" / "test_spelling.py").write_text(source.replace('"celcius, fahrenheit"', '"celsius, fahrenheit"'))

run_pytest("tests/test_spelling.py", "-q")


..                                                                       [100%]
2 passed


### AttributeError: 'tuple' object has no attribute 'strip'


In [35]:
%%writefile scratch/stations/tests/test_single.py
import pytest

from readings import parse_reading


@pytest.mark.parametrize("line", [("Bergen,4.2",), ("Oslo,-2.4",)])
def test_a_reading_has_a_temperature(line):
    station, celsius = parse_reading(line)
    assert celsius is not None


Writing scratch/stations/tests/test_single.py


In [36]:
run_pytest("tests/test_single.py", "-q")


FF                                                                       [100%]
=================================== FAILURES ===================================
___________________ test_a_reading_has_a_temperature[line0] ____________________

line = ('Bergen,4.2',)

    @pytest.mark.parametrize("line", [("Bergen,4.2",), ("Oslo,-2.4",)])
    def test_a_reading_has_a_temperature(line):
>       station, celsius = parse_reading(line)

tests/test_single.py:8: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 

line = ('Bergen,4.2',)

    def parse_reading(line):
        """A (station, celsius) pair from a line such as 'Bergen,4.2'. An empty reading is None.
    
        A minus sign written as U+2212, as some spreadsheets write it, reads as a hyphen-minus.
        """
>       station, celsius = line.strip().split(",")
E       AttributeError: 'tuple' object has no attribute 'strip'

readings.py:11: AttributeError
___________________ test_a_reading_has_a_tempera

With one name, every item in the list is that argument's whole value, so `line` received the tuple
`('Bergen,4.2',)`, as `line = ('Bergen,4.2',)` in the report shows, and the IDs fell back to `line0`
and `line1`. A tuple is for several names. For a single name, list the values themselves:


In [37]:
path = PROJECT / "tests" / "test_single.py"
path.write_text(path.read_text().replace('[("Bergen,4.2",), ("Oslo,-2.4",)]', '["Bergen,4.2", "Oslo,-2.4"]'))

run_pytest("tests/test_single.py", "-v")


============================= test session starts ==============================
collecting ... collected 2 items

tests/test_single.py::test_a_reading_has_a_temperature[Bergen,4.2] PASSED [ 50%]
tests/test_single.py::test_a_reading_has_a_temperature[Oslo,-2.4] PASSED [100%]

============================== 2 passed ===============================


### assert 30 == 32 ± 3.2e-05


In [38]:
%%writefile scratch/stations/tests/test_rule_of_thumb.py
import pytest


def rough_fahrenheit(celsius):
    """A rule of thumb: double it, and add 30."""
    return celsius * 2 + 30


def test_the_rule_of_thumb():
    for celsius, fahrenheit in [(0, 32), (10, 50), (37, 98.6), (100, 212)]:
        assert rough_fahrenheit(celsius) == pytest.approx(fahrenheit)


Writing scratch/stations/tests/test_rule_of_thumb.py


In [39]:
run_pytest("tests/test_rule_of_thumb.py", "-q")


F                                                                        [100%]
=================================== FAILURES ===================================
____________________________ test_the_rule_of_thumb ____________________________

    def test_the_rule_of_thumb():
        for celsius, fahrenheit in [(0, 32), (10, 50), (37, 98.6), (100, 212)]:
>           assert rough_fahrenheit(celsius) == pytest.approx(fahrenheit)
E           assert 30 == 32 ± 3.2e-05
E             
E             comparison failed
E             Obtained: 30
E             Expected: 32 ± 3.2e-05

tests/test_rule_of_thumb.py:11: AssertionError
=========================== short test summary info ============================
FAILED tests/test_rule_of_thumb.py::test_the_rule_of_thumb - assert 30 == 32 ...
1 failed


The loop stopped at its first temperature, so the report says the rule is wrong at 0 °C, and nothing
about 10, 37 or 100. Whether the rule works anywhere stays unknown. The same four cases,
parametrized:


In [40]:
%%writefile scratch/stations/tests/test_rule_of_thumb.py
import pytest


def rough_fahrenheit(celsius):
    """A rule of thumb: double it, and add 30."""
    return celsius * 2 + 30


@pytest.mark.parametrize("celsius, fahrenheit", [(0, 32), (10, 50), (37, 98.6), (100, 212)])
def test_the_rule_of_thumb(celsius, fahrenheit):
    assert rough_fahrenheit(celsius) == pytest.approx(fahrenheit)


Overwriting scratch/stations/tests/test_rule_of_thumb.py


In [41]:
run_pytest("tests/test_rule_of_thumb.py", "-q", "--tb=no")


F.FF                                                                     [100%]
=========================== short test summary info ============================
FAILED tests/test_rule_of_thumb.py::test_the_rule_of_thumb[0-32] - assert 30 ...
FAILED tests/test_rule_of_thumb.py::test_the_rule_of_thumb[37-98.6] - assert ...
FAILED tests/test_rule_of_thumb.py::test_the_rule_of_thumb[100-212] - assert ...
3 failed, 1 passed


Now the report shows the whole picture: the rule is exact at 10 °C and wrong at the three other
temperatures, each named by its values.

Last, the notebook is finished with its files, so this cell removes the scratch folder, with the
module and every test file:


In [42]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- `@pytest.mark.parametrize("a, b", [(1, 2), (3, 4)])` runs a test once for every set of values, and
  pytest reports every run as a test of its own.
- A test ID in brackets names every case: built from the values when they are short, and from `ids`
  or `pytest.param(..., id=...)` when they are not.
- A case's node ID runs it alone, `-k` matches part of an ID, and `--lf` runs only the cases that
  failed.
- `pytest.param(..., marks=pytest.mark.xfail(reason=...))` records a case that is expected to fail.
- Stacked `parametrize` decorators run every combination, and the count multiplies.
- A fixture with `params` runs every test that requests it once for each value, read from
  `request.param`.
- A loop in a test stops at its first failing case, so parametrize a table instead.


## What is next

The **Testing Failure** notebook turns to the cases where the right answer is an exception: tests
that pass only when the code raises the right error, with the right message.


---

&#8592; **Previous:** [Fixtures](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/testing-and-packaging/04-fixtures.ipynb)  &nbsp;·&nbsp;  [Testing and Packaging Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/testing-and-packaging.html)
